# VoiceArm

A robot arm you can talk to in Tamil or English. Say something like
"sivappu block-ah bowl-la vai" and a simulated Franka Panda picks up the red
block and puts it in the bowl.

[Source on GitHub](https://github.com/dheepakkaran/VoiceArm-Bilingual-Tamil-English-Voice-Controlled-Robotic-Arm-MuJoCo)

**Turn on the GPU first:** Runtime → Change runtime type → `T4 GPU`.
Simulation only.

## Setup

MuJoCo renders off-screen, so it needs an EGL driver.

In [ ]:
import subprocess
subprocess.run("apt-get -qq update", shell=True)
subprocess.run("apt-get -qq install -y libegl1 libgles2 libosmesa6 > /dev/null",
               shell=True)
print("graphics libraries installed")

In [ ]:
import os, subprocess, sys
from pathlib import Path

PROJ = Path("/content/voicearm")
if not PROJ.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dheepakkaran/VoiceArm-Bilingual-Tamil-English-Voice-Controlled-Robotic-Arm-MuJoCo.git", str(PROJ)], check=True)

os.chdir(PROJ)
sys.path.insert(0, str(PROJ))
print("project at", PROJ)

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -2
print("dependencies installed")

In [ ]:
# Just the Franka Panda, not all of mujoco_menagerie.
import json, urllib.request
from pathlib import Path

DIR = Path("assets/mujoco_menagerie/franka_emika_panda")
BASE = ("https://raw.githubusercontent.com/google-deepmind/"
        "mujoco_menagerie/main/franka_emika_panda")
(DIR / "assets").mkdir(parents=True, exist_ok=True)

for name in ["panda.xml", "hand.xml", "scene.xml", "LICENSE"]:
    urllib.request.urlretrieve(f"{BASE}/{name}", DIR / name)

api = ("https://api.github.com/repos/google-deepmind/mujoco_menagerie/"
       "contents/franka_emika_panda/assets")
for mesh in json.load(urllib.request.urlopen(api)):
    if mesh["type"] == "file":
        urllib.request.urlretrieve(mesh["download_url"], DIR / "assets" / mesh["name"])

print(len(list((DIR / "assets").iterdir())), "mesh files")

In [ ]:
import torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## Arm, kinematics and grasping

No models yet -- this is physics and inverse kinematics only, so it runs in a
few seconds.

In [ ]:
!python scripts/demo.py --check

## The whole pipeline

Downloads Whisper, Qwen2.5-1.5B and OWLv2, about 6 GB, then runs three commands
end to end: English, Tanglish, and Tamil script.

In [ ]:
!python scripts/demo.py

In [ ]:
!python scripts/demo.py --text "neela block-ah bowl-la podu"

## Talk to it

Starts the web app with a public link. Record yourself in Tamil, English or a
mix, or just type. The link works while this notebook session is alive.

In [ ]:
import importlib.util, sys
sys.path.insert(0, ".")

spec = importlib.util.spec_from_file_location("voicearm_app", "webapp/app.py")
app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app)

app.build().queue(max_size=8).launch(share=True)